# Data enrichment — Wikipedia

Selection: top 10 per year and genre by numVotes descending, then averageRating descending, then IMDb ID ascending; unrated films rank last.

Owner: Hugo. Run from top to bottom after notebook 01. All cohort selection,
cleaning, identity checks, joins and summaries run in Spark. Only HTTP collection
and its persistent response cache use ordinary Python. Text is copied from
Wikipedia introductions; no synopsis or biography is generated.


In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / "jobs"), str(ROOT)]
from pyspark.sql import functions as F, Window
from pyspark.sql.types import StringType
from urllib.parse import unquote
from common import get_spark, read_silver, DATA_DIR
from enrichment.wikipedia import Collector
from enrichment.spark_helpers import top_films, principal_candidates, choose_principals, export_csv
spark = get_spark("imdb_data_enrichment", driver_memory="2g", cores=2)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version, "Python:", sys.version.split()[0])
spark.conf.set("spark.sql.shuffle.partitions", "100")
FIRST_YEAR, LAST_YEAR, TOP_N = 1980, 2025, 10
PRINCIPALS_PER_FILM = 3
MAX_DRIVER_IDS = 250_000
RAW = DATA_DIR / "enrichment" / "raw"
WORK = DATA_DIR / "enrichment" / "tables"
RESULTS = ROOT / "results" / "enrichment"
WORK.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

def materialize(df, name):
    path = WORK / name
    df.write.mode("overwrite").parquet(str(path))
    return spark.read.parquet(str(path))

def bounded_values(df, column):
    rows = df.select(column).distinct().orderBy(column).limit(MAX_DRIVER_IDS + 1).collect()
    if len(rows) > MAX_DRIVER_IDS:
        raise RuntimeError("Selected cohort exceeds the configured driver collection bound")
    return [r[column] for r in rows]


Spark: 4.3.0 Python: 3.13.18


## 1. Top ten feature-length films per genre and release year
Popularity is the current `numVotes`, then `averageRating`, then ascending IMDb ID.
Unrated films rank last and remain eligible when a group has fewer rated films.
A multi-genre film appears in each relevant group. Years are inclusive; feature
length uses the shared silver flag. Principals are distinct people: actors,
actresses and directors have priority, followed by other credited principals.
All candidates are checked so another sourced principal can replace a missing biography.
Two sourced IMDb IDs pointing to the same Wikidata person count as one biography.


In [2]:
titles = read_silver(spark, "title.basics")
ratings = read_silver(spark, "title.ratings")
selected = materialize(top_films(titles, ratings, FIRST_YEAR, LAST_YEAR, TOP_N), "selected")
films = materialize(selected.select("tconst", "primaryTitle", "startYear", "numVotes", "averageRating").distinct(), "films")
assert films.count() > 0, "The selected IMDb cohort is empty"
candidates = materialize(principal_candidates(read_silver(spark, "title.principals"), films)
    .join(read_silver(spark, "name.basics").select("nconst", "primaryName"), "nconst", "left"), "candidates")
targets = materialize(films.select(F.col("tconst").alias("imdb_id"))
    .union(candidates.select(F.col("nconst").alias("imdb_id"))).distinct(), "targets")
print("Genre-year memberships:", selected.count(), "Unique films:", films.count(), "Target entities:", targets.count())
selected.orderBy("startYear", "genre", "genre_rank").show(20, truncate=False)
export_csv(selected.orderBy("startYear", "genre", "genre_rank"), RESULTS / "selected_films.csv")


Genre-year memberships: 10543 Unique films: 7526 Target entities: 76133
+---------+--------------------------------+---------+--------+-------------+------+----------+
|tconst   |primaryTitle                    |startYear|numVotes|averageRating|genre |genre_rank|
+---------+--------------------------------+---------+--------+-------------+------+----------+
|tt0081573|Superman II                     |1980     |125403  |6.8          |Action|1         |
|tt0080745|Flash Gordon                    |1980     |67402   |6.5          |Action|2         |
|tt0080736|The Final Countdown             |1980     |30643   |6.7          |Action|3         |
|tt0080377|Any Which Way You Can           |1980     |22915   |6.1          |Action|4         |
|tt0081529|Smokey and the Bandit II        |1980     |19460   |5.3          |Action|5         |
|tt0080472|Bronco Billy                    |1980     |14025   |6.2          |Action|6         |
|tt0081506|Shogun Assassin                 |1980     |12211   |7

## 2. Human-authored sources and identity evidence
Wikidata's exact IMDb identifier (`P345`) establishes identity. Multiple Wikidata
entities for one IMDb ID are rejected. Only an English Wikipedia sitelink is used;
there is no title/name fuzzy match. Wikipedia's `wikibase_item` page property must
also match that Wikidata entity. Disambiguation pages and empty extracts are rejected.
Wikipedia introductions serve as short film summaries and biographies, not necessarily
full plot accounts. Text provenance is Wikipedia; individual edits are not independently
audited for authorship. Source URLs, revision links, retrieval timestamps, license
and article-history attribution accompany every retained text.


In [3]:
collector = Collector(RAW)
ids = bounded_values(targets, "imdb_id")
identity_paths = collector.identities(ids)
identity_schema = "retrieved_at string, request_url string, response struct<results:struct<bindings:array<struct<imdb_id:struct<value:string>,item:struct<value:string>,article:struct<value:string>>>>>"
raw_identities = spark.read.schema(identity_schema).option("mode", "FAILFAST").json(identity_paths)
bindings = (raw_identities.select("retrieved_at", "request_url", F.explode("response.results.bindings").alias("b"))
    .select(F.col("b.imdb_id.value").alias("imdb_id"), F.col("b.item.value").alias("wikidata_url"),
            F.col("b.article.value").alias("article_url"), "retrieved_at", "request_url"))
identity_groups = bindings.groupBy("imdb_id").agg(
    F.countDistinct("wikidata_url").alias("identity_count"),
    F.min("wikidata_url").alias("wikidata_url"), F.min("article_url").alias("article_url"),
    F.max("retrieved_at").alias("identity_retrieved_at"), F.min("request_url").alias("identity_query_url"))
identities = materialize(targets.join(identity_groups, "imdb_id", "left")
    .withColumn("identity_count", F.coalesce("identity_count", F.lit(0)))
    .withColumn("qid", F.regexp_extract("wikidata_url", r"(Q[0-9]+)$", 1))
    .withColumn("identity_status", F.when(F.col("identity_count") == 0, "no_wikidata_match")
                .when(F.col("identity_count") > 1, "ambiguous_imdb_id")
                .when(F.col("article_url").isNull(), "no_english_article").otherwise("matched")), "identities")
identities.groupBy("identity_status").count().show(truncate=False)


Wikidata: 100/76133 IDs


Wikidata: 200/76133 IDs


Wikidata: 300/76133 IDs


Wikidata: 400/76133 IDs


Wikidata: 500/76133 IDs


Wikidata: 600/76133 IDs


... (753 outputs left out)

Wikidata: 76000/76133 IDs


Wikidata: 76100/76133 IDs


Wikidata: 76133/76133 IDs


+------------------+-----+
|identity_status   |count|
+------------------+-----+
|no_wikidata_match |25232|
|ambiguous_imdb_id |98   |
|matched           |38072|
|no_english_article|12731|
+------------------+-----+



## 3. Batched, cached API collection and Spark cleaning
Requests are sequential, at least 1.5 seconds apart, with an explicit User-Agent,
`maxlag=5` for Wikipedia, bounded retries, exponential backoff and `Retry-After`.
Wikidata batches contain 100 IDs; Wikipedia batches contain at most 20 extracts.
New Wikidata requests bind the ID list before the property/sitelink joins using
a query-plan hint. Earlier successful responses are reused without refetching.
Successful raw responses are saved atomically under ignored `data/enrichment/raw/`.
A failed request stops execution rather than being reported as missing source data.
Rerunning reuses successful responses. Only paths from this run enter the analysis.


In [4]:
decode_title = F.udf(lambda value: unquote(value).replace("_", " ") if value else None, StringType())
page_titles = bounded_values(identities.filter("identity_status = 'matched'")
    .select(decode_title(F.regexp_replace("article_url", r"^https://en.wikipedia.org/wiki/", "")).alias("title")), "title")
page_paths = collector.extracts(page_titles)
page_schema = "retrieved_at string, request_url string, response struct<query:struct<pages:array<struct<pageid:bigint,ns:int,title:string,extract:string,fullurl:string,lastrevid:bigint,pageprops:struct<wikibase_item:string,disambiguation:string>>>>>"
raw_pages = (spark.read.schema(page_schema).option("mode", "FAILFAST").json(page_paths) if page_paths
             else spark.createDataFrame([], page_schema))
pages = (raw_pages.select("retrieved_at", "request_url", F.explode("response.query.pages").alias("p"))
    .filter((F.col("p.ns") == 0) & F.col("p.pageprops.disambiguation").isNull())
    .select(F.col("p.pageprops.wikibase_item").alias("qid"), F.col("p.extract").alias("source_text"),
            F.col("p.fullurl").alias("source_url"), F.col("p.lastrevid").alias("revision_id"),
            F.col("p.title").alias("page_title"), "retrieved_at", F.col("request_url").alias("text_query_url"))
    .withColumn("text", F.trim(F.regexp_replace("source_text", r"\s+", " ")))
    .filter(F.col("qid").isNotNull() & (F.length("text") > 0) & F.col("source_url").isNotNull()))
latest = Window.partitionBy("qid").orderBy(F.desc("retrieved_at"), F.desc("revision_id"), "source_url")
pages = pages.withColumn("duplicate_rank", F.row_number().over(latest)).filter("duplicate_rank = 1").drop("duplicate_rank", "source_text")
entities = materialize(identities.join(pages, "qid", "left")
    .withColumn("has_text", (F.col("identity_status") == "matched") & F.col("text").isNotNull())
    .withColumn("text", F.when(F.col("has_text"), F.col("text")))
    .withColumn("lineage", F.when(F.col("has_text"), F.col("source_url")))
    .withColumn("source_language", F.when(F.col("has_text"), F.lit("en")))
    .withColumn("source_license", F.when(F.col("has_text"), F.lit("CC BY-SA 4.0; see article for additional terms")))
    .withColumn("license_url", F.when(F.col("has_text"), F.lit("https://creativecommons.org/licenses/by-sa/4.0/")))
    .withColumn("attribution_url", F.when(F.col("has_text"), F.concat("source_url", F.lit("?action=history"))))
    .withColumn("revision_url", F.when(F.col("has_text"), F.concat(F.lit("https://en.wikipedia.org/w/index.php?oldid="), F.col("revision_id"))))
    .withColumn("match_evidence", F.concat(F.lit("Wikidata P345="), "imdb_id", F.lit("; entity="), F.coalesce("qid", F.lit("missing")),
                                         F.when(F.col("has_text"), F.lit("; Wikipedia wikibase_item verified")).otherwise(F.lit("; no text accepted"))))
    .withColumn("note", F.when(F.col("has_text"), F.lit("Wikipedia introduction; whitespace normalized"))
                .when(F.col("identity_status") != "matched", F.col("identity_status"))
                .otherwise(F.lit("no_nonempty_verified_article_extract"))), "entities")
print("HTTP requests:", collector.requests, "Cached requests:", collector.cache_hits)
entities.groupBy("note").count().orderBy(F.desc("count")).show(truncate=False)


Wikipedia: 20/38069 pages


Wikipedia: 40/38069 pages


Wikipedia: 60/38069 pages


Wikipedia: 80/38069 pages


Wikipedia: 100/38069 pages


Wikipedia: 120/38069 pages


... (1,895 outputs left out)

Wikipedia: 38040/38069 pages


Wikipedia: 38060/38069 pages


Wikipedia: 38069/38069 pages


HTTP requests: 1732 Cached requests: 934
+---------------------------------------------+-----+
|note                                         |count|
+---------------------------------------------+-----+
|Wikipedia introduction; whitespace normalized|37769|
|no_wikidata_match                            |25232|
|no_english_article                           |12731|
|no_nonempty_verified_article_extract         |303  |
|ambiguous_imdb_id                            |98   |
+---------------------------------------------+-----+



## 4. Joins, complete-case counts and reusable exports
At most three distinct sourced principals are selected per film, with missing
candidates retained if fewer are available. Counts include every selected film,
even films with no principal credits. Full text CSVs are written locally in
`results/enrichment/` and excluded from Git; compact coverage and lineage samples
are versioned. Complete Spark tables remain under `data/enrichment/tables/`.


In [5]:
film_text = materialize(films.join(entities.withColumnRenamed("imdb_id", "tconst"), "tconst", "left")
    .withColumnRenamed("text", "synopsis").withColumn("synopsis_kind", F.lit("Wikipedia introduction")), "film_synopses")
all_bios = candidates.join(entities.withColumnRenamed("imdb_id", "nconst"), "nconst", "left")
principal_text = materialize(choose_principals(all_bios, PRINCIPALS_PER_FILM).withColumnRenamed("text", "biography"), "principal_biographies")
enriched_memberships = materialize(selected.join(
    film_text.select("tconst", "synopsis", "synopsis_kind", "has_text", "lineage", "source_language", "retrieved_at", "revision_url", "source_license", "license_url", "attribution_url", "match_evidence", "note"),
    "tconst", "left"), "enriched_film_genres")
counts = principal_text.groupBy("tconst").agg(F.countDistinct("nconst").alias("selected_principals"),
    F.countDistinct(F.when(F.col("has_text"), F.col("nconst"))).alias("sourced_principals"))
available = candidates.groupBy("tconst").agg(F.countDistinct("nconst").alias("candidate_principals"))
coverage = materialize(film_text.select("tconst", "primaryTitle", "startYear", F.col("has_text").alias("has_synopsis"))
    .join(counts, "tconst", "left").join(available, "tconst", "left")
    .fillna(0, subset=["selected_principals", "sourced_principals", "candidate_principals"])
    .withColumn("has_three_biographies", F.col("sourced_principals") >= PRINCIPALS_PER_FILM)
    .withColumn("complete", F.col("has_synopsis") & F.col("has_three_biographies"))
    .withColumn("biography_note", F.when(F.col("has_three_biographies"), "requirement_met")
                .when(F.col("candidate_principals") < PRINCIPALS_PER_FILM, "fewer_than_three_distinct_principals_in_silver")
                .otherwise("fewer_than_three_verified_english_biographies")), "film_coverage")
summary = coverage.agg(F.count("*").alias("films"), F.sum(F.col("has_synopsis").cast("int")).alias("films_with_synopsis"),
    F.sum(F.col("has_three_biographies").cast("int")).alias("films_with_three_biographies"),
    F.sum((~F.col("has_three_biographies")).cast("int")).alias("films_short_of_three_biographies"),
    F.sum(F.col("complete").cast("int")).alias("complete_films"))
by_group = selected.join(coverage.select("tconst", "has_synopsis", "has_three_biographies", "complete"), "tconst")
by_group = by_group.groupBy("startYear", "genre").agg(F.count("*").alias("selected_films"),
    F.sum(F.col("has_synopsis").cast("int")).alias("synopses"),
    F.sum(F.col("has_three_biographies").cast("int")).alias("films_with_three_biographies"),
    F.sum(F.col("complete").cast("int")).alias("complete_films"))
# Retain zero-coverage year/genre combinations as well as observed groups.
years = spark.range(FIRST_YEAR, LAST_YEAR + 1).select(F.col("id").cast("int").alias("startYear"))
genres = titles.filter("is_feature_film").select(F.explode("genres").alias("genre")).distinct().filter(F.col("genre").isNotNull())
by_group = years.crossJoin(genres).join(by_group, ["startYear", "genre"], "left").fillna(0)
summary.show(truncate=False)
coverage.groupBy("biography_note").count().show(truncate=False)
by_group.orderBy("startYear", "genre").show(30, truncate=False)
for frame, filename in [(summary, "coverage_summary"), (coverage.orderBy("tconst"), "film_coverage"),
                        (by_group.orderBy("startYear", "genre"), "coverage_by_year_genre"),
                        (entities.groupBy("note").count().orderBy("note"), "source_status_counts"),
                        (film_text.orderBy("tconst"), "film_synopses"),
                        (enriched_memberships.orderBy("startYear", "genre", "genre_rank"), "enriched_film_genres"),
                        (principal_text.orderBy("tconst", "principal_rank"), "principal_biographies")]:
    export_csv(frame, RESULTS / (filename + ".csv"))
export_csv(entities.orderBy("imdb_id").limit(100).drop("text", "identity_query_url", "text_query_url"), RESULTS / "lineage_sample.csv")

export_csv(film_text.filter("has_text").orderBy("tconst").limit(20)
    .select("tconst", "primaryTitle", "synopsis", "lineage", "source_language", "retrieved_at", "revision_url", "source_license", "license_url", "attribution_url", "match_evidence", "note"), RESULTS / "synopsis_sample.csv")
export_csv(principal_text.filter("has_text").orderBy("tconst", "principal_rank").limit(60)
    .select("tconst", "nconst", "primaryName", "biography", "lineage", "source_language", "retrieved_at", "revision_url", "source_license", "license_url", "attribution_url", "match_evidence", "note"), RESULTS / "biography_sample.csv")


+-----+-------------------+----------------------------+--------------------------------+--------------+
|films|films_with_synopsis|films_with_three_biographies|films_short_of_three_biographies|complete_films|
+-----+-------------------+----------------------------+--------------------------------+--------------+
|7526 |6340               |6671                        |855                             |6179          |
+-----+-------------------+----------------------------+--------------------------------+--------------+

+----------------------------------------------+-----+
|biography_note                                |count|
+----------------------------------------------+-----+
|fewer_than_three_distinct_principals_in_silver|167  |
|fewer_than_three_verified_english_biographies |688  |
|requirement_met                               |6671 |
+----------------------------------------------+-----+



+---------+-----------+--------------+--------+----------------------------+--------------+
|startYear|genre      |selected_films|synopses|films_with_three_biographies|complete_films|
+---------+-----------+--------------+--------+----------------------------+--------------+
|1980     |Action     |10            |10      |10                          |10            |
|1980     |Adult      |10            |4       |8                           |4             |
|1980     |Adventure  |10            |10      |10                          |10            |
|1980     |Animation  |10            |8       |9                           |7             |
|1980     |Biography  |10            |10      |10                          |10            |
|1980     |Comedy     |10            |10      |10                          |10            |
|1980     |Crime      |10            |10      |10                          |10            |
|1980     |Documentary|10            |10      |10                          |10  

In [6]:
assert enriched_memberships.count() == selected.count()
assert coverage.count() == films.count()
assert selected.groupBy("startYear", "genre").count().filter(F.col("count") > TOP_N).count() == 0
assert selected.groupBy("tconst", "startYear", "genre").count().filter("count > 1").count() == 0
assert entities.count() == targets.count()
assert entities.filter(F.col("has_text") & (F.col("lineage").isNull() | F.col("retrieved_at").isNull())).count() == 0
assert principal_text.filter("has_text").groupBy("tconst", "qid").count().filter("count > 1").count() == 0
assert principal_text.groupBy("tconst", "nconst").count().filter("count > 1").count() == 0
assert coverage.filter(F.col("sourced_principals") > PRINCIPALS_PER_FILM).count() == 0
film_text.filter("has_text").select("tconst", "primaryTitle", "lineage", "revision_url", F.substring("synopsis", 1, 180).alias("text_preview")).show(5, truncate=False)
principal_text.filter("has_text").select("tconst", "nconst", "primaryName", "lineage").show(10, truncate=False)
print("Full cohort validation passed. Missing-source counts are in coverage_summary.csv.")
spark.stop()


+---------+------------------+---------------------------------------------------+-----------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|tconst   |primaryTitle      |lineage                                            |revision_url                                         |text_preview                                                                                                                                                                                  |
+---------+------------------+---------------------------------------------------+-----------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|tt0061876|The C

+---------+---------+--------------------+-------------------------------------------------+
|tconst   |nconst   |primaryName         |lineage                                          |
+---------+---------+--------------------+-------------------------------------------------+
|tt0061876|nm0603314|Nonna Mordyukova    |https://en.wikipedia.org/wiki/Nonna_Mordyukova   |
|tt0061876|nm0125909|Rolan Bykov         |https://en.wikipedia.org/wiki/Rolan_Bykov        |
|tt0061876|nm0623970|Raisa Nedashkovskaya|https://en.wikipedia.org/wiki/Rayisa_Nedashkivska|
|tt0078349|nm0847580|Keiko Takeshita     |https://en.wikipedia.org/wiki/Keiko_Takeshita    |
|tt0078349|nm0793336|Tarô Shigaki        |https://en.wikipedia.org/wiki/Taro_Shigaki       |
|tt0078349|nm0463514|Asao Koike          |https://en.wikipedia.org/wiki/Asao_Koike         |
|tt0078780|nm0705717|Gilda Radner        |https://en.wikipedia.org/wiki/Gilda_Radner       |
|tt0078780|nm0000345|Billy Crystal       |https://en.wikipedia.org/wik